In [9]:
# ======================================================
# 📊 CSV( extracted_response, expected_output ) → 평가 자동화
# ======================================================
# 필요 패키지:
#   pip install -U pandas evaluate bert-score sentence-transformers hf_transfer absl-py rouge-score nltk
# ======================================================

import pandas as pd
import numpy as np
import json, re, datetime
import evaluate
from sentence_transformers import SentenceTransformer, util

# -----------------------------
# 전처리 함수
# -----------------------------
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.replace("\u200b", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -----------------------------
# CSV 로드
# -----------------------------
def load_pairs(csv_path: str):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="utf-8")

    if not {"extracted_response", "expected_output"}.issubset(df.columns):
        raise ValueError("CSV에 'extracted_response', 'expected_output' 컬럼이 필요합니다.")

    preds = df["extracted_response"].fillna("").map(normalize_text).tolist()
    refs = df["expected_output"].fillna("").map(normalize_text).tolist()
    return preds, refs, df

# -----------------------------
# 지표 계산
# -----------------------------
def compute_metrics(preds, refs):
    print("평가지표 계산 중...")

    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load("meteor")
    bertscore = evaluate.load("bertscore")

    # BLEU / ROUGE / METEOR
    bleu_res = bleu.compute(predictions=preds, references=refs)
    rouge_res = rouge.compute(predictions=preds, references=refs)
    meteor_res = meteor.compute(predictions=preds, references=refs)

    # BERTScore
    bert_res = bertscore.compute(predictions=preds, references=refs, lang="ko")
    bert_f1 = np.mean(bert_res["f1"])

    # Embedding Cosine (의미 유사도)
    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    emb_pred = model.encode(preds, convert_to_tensor=True, show_progress_bar=True)
    emb_ref = model.encode(refs, convert_to_tensor=True, show_progress_bar=True)
    cosine_scores = util.cos_sim(emb_pred, emb_ref).diagonal().cpu().numpy().tolist()
    emb_cos_mean = float(np.mean(cosine_scores))

    result = {
        "bleu": float(bleu_res["bleu"]),
        "rougeL": float(rouge_res["rougeL"]),
        "meteor": float(meteor_res["meteor"]),
        "bertscore_f1_mean": float(bert_f1),
        "embedding_cosine_mean": float(emb_cos_mean),
        "bertscore_f1_each": [float(x) for x in bert_res["f1"]],
        "embedding_cosine_each": cosine_scores,
    }
    return result

# -----------------------------
# 결과 저장
# -----------------------------
def save_results(df, metrics):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # 요약 JSON
    summary = {
        "BLEU": metrics["bleu"],
        "ROUGE-L": metrics["rougeL"],
        "METEOR": metrics["meteor"],
        "BERTScore(F1_mean)": metrics["bertscore_f1_mean"],
        "Embedding_Cosine_mean": metrics["embedding_cosine_mean"],
    }
    with open(f"metric_summary_{ts}.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # 개별 CSV
    df_out = df.copy()
    df_out["BERTScore_F1"] = metrics["bertscore_f1_each"]
    df_out["Embedding_Cosine"] = metrics["embedding_cosine_each"]
    df_out.to_csv(f"metric_detailed_{ts}.csv", index=False, encoding="utf-8-sig")

    print("\n=== 📈 요약 결과 ===")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"\n✅ 저장 완료\n- 요약: metric_summary_{ts}.json\n- 세부결과: metric_detailed_{ts}.csv")

# -----------------------------
# 실행부
# -----------------------------
if __name__ == "__main__":
    csv_path = "./qwen_eval_results_min_preprocessed_20251021_184940.csv"
    preds, refs, df = load_pairs(csv_path)
    metrics = compute_metrics(preds, refs)
    save_results(df, metrics)

평가지표 계산 중...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


=== 📈 요약 결과 ===
{
  "BLEU": 0.004048775338862849,
  "ROUGE-L": 0.08090174899353642,
  "METEOR": 0.0669624063948609,
  "BERTScore(F1_mean)": 0.6726641443040636,
  "Embedding_Cosine_mean": 0.6429074117430934
}

✅ 저장 완료
- 요약: metric_summary_20251021_203952.json
- 세부결과: metric_detailed_20251021_203952.csv
